# 9-Class 모델 코드 검토 (2026-09-04, 손은총)

오늘 비교·하이브리드 실험에 쓴 모델 정의 코드 — `train9/models.py` 하나에 전부 들어 있다.

| 부분 | 역할 |
|---|---|
| `make()` | 알고리즘 하나(et·xgb·lgbm·hier_*)를 만드는 공장 함수 |
| `HierarchicalNine` | `make()`를 두 번 불러(이진 머리 + 유형 머리) 9-Class 계층 분해를 만드는 클래스 |

`hier_et`/`hier_xgb`/`hier_lgbm` 은 전부 `HierarchicalNine` 인스턴스고, **`family` 인자 하나만 다르다.**
이진 머리·유형 머리 둘 다 그 family 로 통일된다.

In [1]:
import sys, inspect
sys.path.insert(0, '/workspace')
from train9 import models as M
print(M.__file__)

/workspace/train9/models.py


## §1. `make()` — 알고리즘 공장 함수 (소스 그대로)

In [2]:
print(inspect.getsource(M.make))

def make(name: str, n_jobs: int = 16, seed: int = SEED, **over):
    if name == 'lgbm':
        from lightgbm import LGBMClassifier
        kw = dict(objective='multiclass', num_class=9, n_estimators=400,
                  learning_rate=0.05, num_leaves=63, min_child_samples=5,
                  subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
                  reg_lambda=1.0, n_jobs=n_jobs, random_state=seed, verbose=-1)
        kw.update(over)
        return LGBMClassifier(**kw)
    if name == 'xgb':
        from xgboost import XGBClassifier
        kw = dict(objective='multi:softprob', num_class=9, n_estimators=400,
                  learning_rate=0.05, max_depth=6, min_child_weight=1.0,
                  subsample=0.9, colsample_bytree=0.8, reg_lambda=1.0,
                  tree_method='hist', n_jobs=n_jobs, random_state=seed,
                  eval_metric='mlogloss')
        kw.update(over)
        return XGBClassifier(**kw)
    if name == 'et':
        from sklearn.ensemble

### 읽는 법
- `et`(ExtraTrees): `min_samples_leaf=1` — 잎이 순수해질 때까지 쪼갠다. **오늘 하이브리드 버그의 근원**이다.
  잎이 순수하면 확률이 0/1 에 몰리고(실측: ET 확률 중 패턴 8열이 전부 정확히 0인 행이 **99.3%**),
  순위를 매길 때 동점이 대량 발생한다.
- `xgb`/`lgbm`: 부스팅 계열. `n_estimators=400` 이 기본이다(아래 §3 라운드 민감도 참고).
- `hier_*`: 여기로 안 빠지고 `HierarchicalNine(family=...)` 로 위임한다.

## §2. `HierarchicalNine` — 계층 분해 (소스 그대로)

In [3]:
print(inspect.getsource(M.HierarchicalNine))

class HierarchicalNine:
    """9-Class 를 두 단으로 쪼개 학습하되 **출력 계약은 그대로 9-Class** 인 구현.

    왜 쪼개나: 평탄한 9-Class 는 1,176건뿐인 패턴 행을 8개 클래스로 다시 쪼개 학습한다.
    가장 작은 클래스가 train 60건이라 어떤 모델도 안정적으로 못 배운다. 반면
      (a) '패턴인가 아닌가' 이진 머리는 1,176건을 **한 덩어리로** 쓰고,
      (b) '어느 유형인가' 8-way 머리는 패턴 행만 보므로 불균형이 사라진다.

    p(c|x) = p_bin(패턴|x) · p_type(c|x, 패턴)   (c = 0..7)
    p(8|x) = 1 − p_bin(패턴|x)

    회의가 정한 '1차 = 9-Class 다중분류' 계약을 깨지 않는다 — 입력도 전 거래, 출력도 9개 확률이다.
    """

    def __init__(self, family: str = 'et', n_jobs: int = 32, seed: int = SEED, **over):
        self.family, self.n_jobs, self.seed, self.over = family, n_jobs, seed, over
        self.bin_ = None
        self.type_ = None
        self.type_classes_ = None
        self.classes_ = list(range(9))

    def fit(self, X, y, sample_weight=None):
        import numpy as np
        y = np.asarray(y)
        w = None if sample_weight is None else np.asarray(sample_weight)

        # (a) 이진 머리 — 패턴(0~7) vs 클래스 8. 가중은 두 덩어리 기준으로 다시 잡는다.
        

### 읽는 법 — `fit()`
1. **이진 머리**: `y <= 7`(패턴이냐 아니냐)로 라벨을 접어 학습. 패턴 1,176건(Small)을 **한 덩어리**로 쓴다.
2. **유형 머리**: 패턴 행(`m = y <= 7`)만 골라 8-way 로 학습. 정상 300만 건이 안 섞이니 불균형이 사라진다.
3. 두 머리 다 `make(self.family, ...)` 로 만든다 — **family 하나가 두 모델을 동시에 정한다.**

### 읽는 법 — `predict_proba()`
```
p(c|x) = p_bin(패턴|x) · p_type(c|x, 패턴)      c = 0..7
p(8|x) = 1 − p_bin(패턴|x)
```
이진 확률과 유형 확률을 곱해서 9개 확률로 합친다. 회의가 정한 "1차 = 9-Class 다중분류" 계약이
안 깨지는 이유 — 입력도 전 거래, 출력도 여전히 9개 확률이다.

## §3. 라이브 데모 — 작은 합성 데이터로 실제 동작 확인

In [4]:
import numpy as np
rng = np.random.default_rng(0)
n, d = 20_000, 12
X = rng.random((n, d)).astype(np.float32)
# 클래스 8(정상)이 압도적으로 많은 극심한 불균형을 흉내낸다 (실제 데이터도 패턴이 0.04~0.05%)
y = rng.choice(9, n, p=[0.0006]*8 + [1 - 0.0048])
print(f'클래스 분포: 패턴(0~7) {int((y<=7).sum())}건 · 정상(8) {int((y==8).sum())}건')

클래스 분포: 패턴(0~7) 77건 · 정상(8) 19923건


In [5]:
for fam in ('hier_et', 'hier_xgb', 'hier_lgbm'):
    m = M.make(fam, n_jobs=2, n_estimators=15, seed=0).fit(X, y)
    p = m.predict_proba(X[:5])
    print(f'{fam:10} OK · proba shape {p.shape} · 행 합 1 확인: {np.allclose(p.sum(1), 1)}')

hier_et    OK · proba shape (5, 9) · 행 합 1 확인: True


hier_xgb   OK · proba shape (5, 9) · 행 합 1 확인: True
hier_lgbm  OK · proba shape (5, 9) · 행 합 1 확인: True


### §3-1. `min_samples_leaf=1` 이 만드는 동점 — ET 대 부스팅 비교

작은 트리 수로도 ET 의 확률이 얼마나 0/1 에 몰리는지 바로 보인다(실 데이터에서는 이 비율이 99.3%였다).

In [6]:
import pandas as pd
rows = []
for fam in ('hier_et', 'hier_xgb', 'hier_lgbm'):
    m = M.make(fam, n_jobs=2, n_estimators=15, seed=0).fit(X, y)
    p = m.predict_proba(X)
    zero_rows = (p[:, :8] == 0).all(1).mean()
    uniq = len(np.unique(p[:, :8].max(1)))
    rows.append(dict(family=fam, 패턴열_전부0_비율=round(zero_rows, 4), 알림점수_고유값수=uniq))
pd.DataFrame(rows)

,family,패턴열_전부0_비율,알림점수_고유값수
0,hier_et,0.9962,2
1,hier_xgb,0.0000,19983
2,hier_lgbm,0.0000,20000


## §4. 오늘 이 코드로 잰 결과 (요약)

상세 수치·판정은 `최종보고_모델패밀리·하이브리드_손은총_20260904.docx` 참고. 여기서는 이 코드가
그 결과를 어떻게 만들었는지만 잇는다.

| 실험 | 이 노트북의 어느 부분을 썼나 |
|---|---|
| F1 패밀리 비교(§1의 3종) | `make('hier_et'/'hier_xgb'/'hier_lgbm', n_estimators=200)` |
| F2 라운드 민감도 | 같은 함수, `n_estimators` 만 라이브러리 기본값(600/400/400)으로 |
| H1 하이브리드 | `predict_proba()` 로 뽑은 확률 9열을 모델 간에 가중 결합(`blend_hybrid.py`) |

**결론(§1의 `min_samples_leaf=1` 과 직결)**: ET 의 확률이 0/1 에 극단적으로 몰리는 성질이
① 같은 라운드(200) 비교에서는 ET 가 이기는 이유(무작위 분기가 통화 원핫 같은 쓸모없는 열을 걸러냄)이자
② 확률을 순위로 바꿔 하이브리드할 때 동점 처리를 잘못하면 거래 시각이 가짜 신호로 스며드는 이유였다
(`blend_hybrid.py` 의 `to_rank()` — 처음엔 `argsort(kind='stable')` 를 썼다가 `scipy.stats.rankdata`
로 고쳤다).